### Library and model imports (execute just once)

In [ ]:
from utils import list_images, check_marker_metadata_match, read_image
from utils_infection import detect_infected_cells, detect_infection_load
from data_viz import map_df_column_to_labels
from pathlib import Path
import napari
from cellpose import models, core
from skimage.segmentation import clear_border
from skimage.measure import regionprops_table
import pandas as pd
import numpy as np
import apoc

import matplotlib.pyplot as plt

#Check if notebook has GPU access
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")

#Load pre-trained Cellpose-SAM
model = models.CellposeModel(
    gpu=True, 
    pretrained_model="./models/CPSAM_shirik_ft"
)

# Load pretrained Object Classifier for Mycobacterium infection detection
mtb_cl_filename = "./models/Mtb_segmenter.cl"
mtb_segmenter = apoc.ObjectSegmenter(opencl_filename=mtb_cl_filename)

### Define OCs to analyze and their channel position

In [ ]:
# Define the channels you want to analyze using the following structure:
# markers = [(channel_name, channel_nr),(..., ...)]
# Remember in Python one starts counting from 0, so your first channel will be 0
# i.e. markers = [("SD_DAPI", 0), ("SD_RFP", 1)]

# This list hold the markers for signal intensity analysis 
MARKERS = [("SD_BF", 0), ("SD_DAPI", 1), ("SD_AF647", 2)]

# Define brightfield channel position to use as input for CellposeSAM-mediated cell segmentation
BRIGHTFIELD_CHANNEL = 0

# Define Mtb channel position to use as input for APOC ObjectSegmenter
MTB_CHANNEL = 1

### Define the path containing your images

In [ ]:
# Copy the path to the folder containing your images in between the quotation marks
data_folder = r"X:\Shirin\260720_SK0074_iPSDMsWT-NINJ1KO_MtbInf-MOI2-20_NoOpsNoSon\260721_SK_SK0074_Exp01_ConfocalMic\RawData"

# If the path is correct you should see a list of the first 10 images in your folder down below
images = list_images(data_folder, format="nd2")
images[:10]

# Check if the defined marker position match the metadata in the image files
metadata_match = check_marker_metadata_match(images, MARKERS)
if not metadata_match:
    raise RuntimeError(
        "Marker/metadata mismatch detected in one or more files. "
        "Fix markers or input data before continuing."
    )

### Define a single image file for exploration
Just edit the number in between brackets [ ] under <code>img_filepath = images[0]</code> to select the file

In [ ]:
# Extract experiment_id from data folder Path object
experiment_id = Path(data_folder).name

# Open any of the  images to analyze by changing the index number in between brackets []
# index number refers to the position of the file inside the images folder
# i.e. to open the first image [0], second [1] - in Python one starts counting from zero
img_filepath = images[1]
img, filename = read_image(img_filepath)

# Perform MIP before feeding the image into the pipeline
img = np.max(img, axis=0)

### Predict cell labels using base CellposeSAM (4.0)

In [ ]:
# Predict cell labels using CellposeSAM from brightfield image
cell_labels, flows, styles = model.eval(img[BRIGHTFIELD_CHANNEL], niter=1000) # need to check the arguments

# Visualize results in Napari
viewer = napari.Viewer(ndisplay=2)
viewer.add_image(img, name=filename)

# Remove cell entities touching the image border
cell_labels = clear_border(cell_labels)
viewer.add_labels(cell_labels, opacity=0.5)

### Detect infected cells

In [ ]:
# Infection detection (same as batch notebook)
infection_stats = []

mtb_labels, infected_cell_labels = detect_infected_cells(img, mtb_segmenter, cell_labels, MTB_CHANNEL, filename, infection_stats)

# Cell labels restricted to Mtb-positive cells (for Napari overlay)
infected_cells_layer = np.where(np.isin(cell_labels, infected_cell_labels), cell_labels, 0).astype(cell_labels.dtype)

In [ ]:
viewer.add_labels(mtb_labels, name="Mtb_labels")
viewer.add_labels(infected_cells_layer, name="infected_cells")

### Extract morphology and intensity features

In [ ]:
# Morphology: computed once (same label_image for all markers)
morphology_properties = [
    "label",
    "area",                          # number of voxels (volume in voxel units)
    "area_bbox",                     # volume of axis-aligned bounding box
    "area_convex",                   # volume of convex hull of the region
    "area_filled",                   # volume after filling holes
    "axis_major_length",             # length of major axis from inertia tensor (elongation)
    "axis_minor_length",             # length of minor axis (second principal axis in 3D)
    "equivalent_diameter_area",      # diameter of sphere with same volume as region
    "euler_number",                  # topology: objects + holes − tunnels (connectivity)
    "extent",                        # volume / bounding-box volume (fill of the box)
    "feret_diameter_max",            # maximum Feret (caliper) diameter
    "solidity",                      # volume / convex-hull volume (compact vs lobed)
    "inertia_tensor_eigvals",        # eigenvalues of inertia tensor (3 values: shape/orientation)
]

# Intensity: computed per marker (different intensity_image each time)
intensity_properties = [
    "label",
    "intensity_mean",
    "intensity_min",
    "intensity_max",
    "intensity_std",
]

In [ ]:
# Compute morphology once (label_image is the same for all markers)
props_morphology = regionprops_table(
    label_image=cell_labels,
    properties=morphology_properties,
)
props_df = pd.DataFrame(props_morphology)

# Loop through markers and extract intensity features only
for marker_name, ch_nr in MARKERS:
    print(f"Analyzing channel: {marker_name} ...")

    props = regionprops_table(
        label_image=cell_labels,
        intensity_image=img[ch_nr],
        properties=intensity_properties,
    )

    intensity_df = pd.DataFrame(props)

    # Rename intensity columns with marker prefix
    prefix = f"{marker_name}"
    rename_map = {"label": "label"}
    for prop in intensity_properties:
        if prop == "label":
            continue
        if prop.startswith("intensity_"):
            suffix = prop.replace("intensity_", "")
            rename_map[prop] = f"{prefix}_{suffix}_int"
    intensity_df.rename(columns=rename_map, inplace=True)

    # Derived columns
    mean_col = rename_map["intensity_mean"]
    max_col = rename_map["intensity_max"]
    # Max / mean ratio (puncta vs diffuse signal)
    intensity_df[f"{prefix}_max_mean_ratio"] = intensity_df[max_col] / intensity_df[mean_col].replace(0, np.nan)

    # Merge intensity data into main dataframe
    props_df = props_df.merge(intensity_df, on="label")
    # Total marker content per cell = mean * area (area from morphology)
    props_df[f"{prefix}_sum_int"] = props_df[mean_col] * props_df["area"]

# Insert the filename as the first column in props_df
props_df.insert(0, "filename", filename)


In [ ]:
# Calculate percentage of cell area occupied by bacteria
props_df = detect_infection_load(mtb_labels, cell_labels, props_df)

# Flag cell label as infected or not
col_idx = props_df.columns.get_loc("label")
props_df.insert(col_idx + 1, "Mtb_infected_cell", props_df["label"].isin(infected_cell_labels))

In [ ]:
props_df

### Visualize extracted features in Napari
Render features per cell for exploratory interpretation.

Input needed from researcher: verify normalization and inspect outliers before analysis.

In [ ]:
# Visualize cell area
map_df_column_to_labels(
    cell_labels,
    props_df,
    value_column="area",
    colormap="inferno",
    visualize=True
)

In [ ]:
# Visualize % bacterial_load
map_df_column_to_labels(
    cell_labels,
    props_df,
    value_column="%_bacterial_load",
    colormap="inferno",
    visualize=True
)